# Biblioteka Streamlit

Streamlit to biblioteka służąca do budowy interaktywnych aplikacji webowych, która pozwala na dynamiczną prezentację wizualizacji danych i wyników analiz bezpośrednio z poziomu kodu Python. Dzięki integracji z Matplotlib, Seaborn czy Bokeh, umożliwia użytkownikowi natychmiastowy wpływ na wygląd i zakres prezentowanych danych poprzez interaktywne widżety.

In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# plus inne dodatkowe potrzebne do przygotowywanych wizualizacji

--------------------
##### ⭐ Zadanie 1:
Przygotuj prostą aplikację w bibliotece **Streamlit**, która wczyta dane z Twojego pliku CSV i wyświetli je w formie tabeli. Następnie umożliw wybór jednej kolumny numerycznej i przedstaw jej rozkład na histogramie. Do utworzenia wykresu wykorzystaj bibliotekę `matplotlib` lub `seaborn`, a następnie wyświetl go za pomocą `st.pyplot()`.

W aplikacji wyświetl również podstawowe informacje o zbiorze:
* liczbę wierszy i kolumn,
* nazwy dostępnych kolumn,
* pierwsze 5 rekordów (podgląd danych).

Zapoznaj się z elementami `st.title`, `st.write`, `st.dataframe`, `st.file_uploader`, `st.selectbox` oraz `st.pyplot` i wykorzystaj je w swoim rozwiązaniu.

In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


st.title("Eksplorator Danych - Zadanie 1")

df = pd.read_csv("dataset.csv", sep=None, engine='python', encoding='utf-8')

st.divider()


st.write("### Informacje o zbiorze danych")
st.write(f"**Liczba wierszy:** {df.shape[0]}")
st.write(f"**Liczba kolumn:** {df.shape[1]}")
st.write("**Dostępne kolumny:**")
st.write(list(df.columns))

st.write("### Podgląd danych")
st.dataframe(df.head())

st.divider()

numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

if not numeric_cols:
    st.warning("W zbiorze nie znaleziono kolumn numerycznych do wizualizacji.")
else:
    selected_col = st.selectbox(
        "Wybierz kolumnę numeryczną do wizualizacji rozkładu:",
        numeric_cols
    )

    if selected_col:
        st.write(f"### Rozkład zmiennej: **{selected_col}**")
        
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.histplot(
            data=df, 
            x=selected_col, 
            bins=30, 
            kde=True, 
            ax=ax, 
            color='#1DB954', 
            edgecolor='black'
        )
        ax.set_title(f"Histogram: {selected_col}", fontsize=14, pad=10)
        ax.set_xlabel(selected_col, fontsize=12)
        ax.set_ylabel("Częstotliwość", fontsize=12)
        
        st.pyplot(fig)# tutaj wpisz swoje rozwiązanie

##### ⭐ Zadanie 2:
Przygotuj aplikację, która umożliwi użytkownikowi interaktywne filtrowanie danych oraz ich wizualizację. Wykorzystaj panel boczny (`st.sidebar`) do umieszczenia tam filtrów. Użytkownik powinien mieć możliwość (jeśli dane są odpowiednie):
* wyboru kolumny kategorialnej i zaznaczenia konkretnych wartości do wyświetlenia (`multiselect`),
* ograniczenia zakresu danych dla wybranej kolumny numerycznej za pomocą suwaka (`slider`).

Po zastosowaniu filtrów aplikacja powinna dynamicznie aktualizować wyświetlaną tabelę oraz wykres (np. `boxplot` lub wykres słupkowy). Zapoznaj się z elementami `st.sidebar`, `st.multiselect`, `st.slider` oraz `st.columns` (do lepszego układu elementów) i wykorzystaj je w swoim rozwiązaniu.

In [ ]:

import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

st.set_page_config(page_title="Zadanie 2: Filtrowanie", layout="wide")

st.title("🎛️ Interaktywny Dashboard Danych")

@st.cache_data
def load_data():
    return pd.read_csv("dataset.csv", sep=None, engine='python', encoding='utf-8', on_bad_lines='skip')

df = load_data()

st.sidebar.header("⚙️ Ustawienia Filtrów")

cat_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()
num_cols = df.select_dtypes(include=['number']).columns.tolist()

if cat_cols:
    selected_cat_col = st.sidebar.selectbox("1. Wybierz kolumnę kategoryczną:", cat_cols)
    unique_vals = sorted(df[selected_cat_col].dropna().unique().tolist())
    
    default_vals = unique_vals[:3] if len(unique_vals) > 3 else unique_vals
    selected_vals = st.sidebar.multiselect(f"Wybierz wartości dla '{selected_cat_col}':", unique_vals, default=default_vals)
else:
    selected_vals = []
    st.sidebar.warning("Brak kolumn tekstowych/kategorycznych w zbiorze.")

if num_cols:
    selected_num_col = st.sidebar.selectbox("2. Wybierz kolumnę numeryczną:", num_cols)
    min_val = float(df[selected_num_col].min())
    max_val = float(df[selected_num_col].max())
    
    if min_val == max_val:
        range_val = (min_val, max_val)
        st.sidebar.info(f"Wszystkie wartości w '{selected_num_col}' są równe: {min_val}")
    else:
        range_val = st.sidebar.slider(f"Zakres dla '{selected_num_col}':", min_val, max_val, (min_val, max_val))
else:
    range_val = None
    st.sidebar.warning("Brak kolumn numerycznych w zbiorze.")

mask = pd.Series(True, index=df.index)

if cat_cols and selected_vals:
    mask = mask & (df[selected_cat_col].isin(selected_vals))

if num_cols and range_val:
    mask = mask & (df[selected_num_col].between(range_val[0], range_val[1]))

df_filtered = df[mask]

st.write("---")
st.subheader(f"Znaleziono rekordów: **{len(df_filtered):,}**")

if not df_filtered.empty:
    col1, col2 = st.columns([3, 2])
    
    with col1:
        st.write("### 📊 Średnie wartości (Wykres słupkowy)")
        fig, ax = plt.subplots(figsize=(10, 6))
        
        sns.barplot(
            data=df_filtered,
            x=selected_cat_col,
            y=selected_num_col,
            palette="viridis",
            ax=ax,
            errorbar=None 
        )
        plt.xticks(rotation=45, ha='right')
        ax.set_title(f"Średnia '{selected_num_col}' dla każdej wartości z '{selected_cat_col}'", pad=15)
        ax.set_xlabel("")
        ax.set_ylabel(f"Średnia {selected_num_col}")
        
        st.pyplot(fig)
        
    with col2:
        st.write("### 📋 Podgląd danych")
        cols_to_show = list(set([selected_cat_col, selected_num_col] + list(df.columns[:3])))
        st.dataframe(df_filtered[cols_to_show], height=400, use_container_width=True)
else:
    st.warning("Brak danych spełniających kryteria filtrów. Zmień ustawienia w panelu bocznym.")

##### ⭐ Zadanie 3:
Przygotuj zaawansowany pulpit nawigacyjny (dashboard), który pozwoli na wielowymiarową analizę danych. Aplikacja powinna umożliwiać:
* wybór dwóch kolumn numerycznych do analizy zależności,
* wybór rodzaju wykresu spośród co najmniej trzech opcji: histogram, wykres punktowy (`scatter plot`), wykres pudełkowy (`boxplot`),
* personalizację wykresu: np. zmianę koloru markerów, wybór kolumny grupującej (`hue`) lub włączenia siatki pomocniczej.

Wykorzystaj `st.tabs`, aby rozdzielić wizualizację od statystycznego podsumowania danych (`df.describe()`). Do tworzenia wykresów możesz wykorzystać bibliotekę `plotly` (`st.plotly_chart`), aby dodać im interaktywności (zoom, tooltips). Zapoznaj się z `st.radio`, `st.checkbox`, `st.tabs` oraz `st.expander`.

In [ ]:

import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Zadanie 3: Dashboard", layout="wide")
st.title("🚀 Zaawansowany Dashboard Analityczny")

@st.cache_data
def load_data():
    return pd.read_csv("dataset.csv", sep=None, engine='python', encoding='utf-8', on_bad_lines='skip')

df = load_data()

num_cols = df.select_dtypes(include=['number']).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()

st.sidebar.header("🛠️ Konfiguracja Dashboardu")

with st.sidebar.expander("1. Wybór Zmiennych", expanded=True):
    val_x = st.selectbox("Zmienna X (Główna):", num_cols, index=0)
    val_y = st.selectbox("Zmienna Y (Tylko dla Scatter Plot):", num_cols, index=1 if len(num_cols) > 1 else 0)
    
    group_col = st.selectbox("Grupowanie (Hue/Kolor):", ["Brak"] + cat_cols)
    grouping = None if group_col == "Brak" else group_col

with st.sidebar.expander("2. Typ i Styl Wykresu", expanded=True):
    chart_type = st.radio("Rodzaj wykresu:", ["Wykres punktowy (Scatter)", "Histogram", "Wykres pudełkowy (Boxplot)"])
    
    base_color = st.color_picker("Wybierz kolor markerów (gdy brak grupowania):", "#1DB954")
    show_grid = st.checkbox("Pokaż siatkę pomocniczą", value=True)

df_plot = df.sample(2000, random_state=42) if len(df) > 2000 else df

tab_viz, tab_stat = st.tabs(["📊 Wizualizacja Danych", "📈 Podsumowanie Statystyczne"])

with tab_viz:
    st.subheader(f"Analiza: {chart_type}")
    
    fig = None
    
    if chart_type == "Wykres punktowy (Scatter)":
        fig = px.scatter(
            df_plot, x=val_x, y=val_y, 
            color=grouping,
            color_discrete_sequence=[base_color] if not grouping else None,
            opacity=0.7,
            title=f"Zależność: {val_x} vs {val_y}"
        )
        
    elif chart_type == "Histogram":
        fig = px.histogram(
            df_plot, x=val_x, 
            color=grouping,
            color_discrete_sequence=[base_color] if not grouping else None,
            nbins=40,
            barmode='overlay',
            title=f"Rozkład zmiennej: {val_x}"
        )
        
    elif chart_type == "Wykres pudełkowy (Boxplot)":
        fig = px.box(
            df_plot, x=grouping, y=val_x, 
            color=grouping,
            color_discrete_sequence=[base_color] if not grouping else None,
            title=f"Wykres pudełkowy: {val_x}" + (f" wg {grouping}" if grouping else "")
        )

    if fig:
        fig.update_xaxes(showgrid=show_grid)
        fig.update_yaxes(showgrid=show_grid)
        st.plotly_chart(fig, use_container_width=True)

with tab_stat:
    st.subheader("Podsumowanie dla wybranych zmiennych numerycznych")
    st.dataframe(df[[val_x, val_y]].describe(), use_container_width=True)
    
    with st.expander("Pokaż macierz korelacji dla wszystkich zmiennych numerycznych"):
        corr = df[num_cols].corr()
        st.dataframe(corr.style.background_gradient(cmap='Greens'))